# Steps
1. Convert token và lemma raw text to csv to get the index
2. Create a list of lemma_POS
3. Extract sentences with lemma_POS in token and lemma csv into individual files for each lemma_POS
4. Parse lại token bằng Stanza
5. Check lại giữa tag cũ và mới xem tỉ lệ sai POS là bao nhiêu

## Import

In [ ]:
import pandas as pd
import os
import stanza
import re
import sys
from pathlib import Path
import shutil

sys.path.append('../data_preprocessing')
from utils import open_txt, save_to_txt, search_in_txt, replace_in_txt, return_stanza_parsed_tags

In [ ]:
def convert_org_stanza(org_lemma_pos):
    # Chuyển từ dạng gốc sang dạng stanza
    org_lemma = org_lemma_pos.rsplit('_')[0]
    org_pos = org_lemma_pos.split('_')[-1]

    stanza_lemma = org_lemma
    if org_pos == 'N':
        stanza_pos = 'NOUN'
    elif org_pos == 'V':
        stanza_pos = 'VERB'
    elif org_pos == 'A':
        stanza_pos = 'ADJ'
    
    return stanza_lemma, stanza_pos

In [ ]:
pilot_folder = Path(f'./SemEval_ger/')
pilot_folder_SemEval = Path(f'./SemEval_ger_SemEval/')

## Convert token and lemma raw text to csv to get the index

In [ ]:
corp_nos = [1, 2]
data_types = ['token', 'lemma']

In [ ]:
for corp_no in corp_nos:
    for data_type in data_types:
        raw_file = f'./semeval2020_ulscd_ger/corpus{corp_no}/{data_type}/ger{corp_no}.txt'
        raw_csv_path = f'./semeval2020_ulscd_ger/corpus{corp_no}/{data_type}/ger{corp_no}.csv'

        # Read txt
        with open(raw_file, 'r') as f:
            lines = f.readlines()
            lines = [line.rstrip('\n') for line in lines]
            # lines = [line.strip() for line in lines if line.strip()]  # Remove empty lines
            # Save to DataFrame and then to CSV
            df = pd.DataFrame(lines)
            df.columns = ['sent']
            df.to_csv(raw_csv_path, index=False, header=True)

In [ ]:
raw_file = './semeval2020_ulscd_ger/corpus1/lemma/ger1.txt'
with open(raw_file, 'r') as f:
    raw_lines = f.readlines()

print("Total lines in txt:", len(raw_lines))
print("Non-empty lines:", sum(1 for l in raw_lines if l.strip()))


## Create a list of lemma_POS

In [ ]:
selected_lemmas = [
    'abbauen_V',
    'abdecken_V',
    'abgebrüht_A',
    'Abgesang_N',
    'Ackergerät_N',
    'Armenhaus_N',
    'artikulieren_V',
    'aufrechterhalten_V',
    'Ausnahmegesetz_N',
    'ausspannen_V',
    'beimischen_V',
    'Dynamik_N',
    'Einreichung_N',
    'Eintagsfliege_N',
    'Engpaß_N',
    'Entscheidung_N',
    'Festspiel_N',
    'Frechheit_N',
    'Fuß_N',
    'Gesichtsausdruck_N',
    'Knotenpunkt_N',
    'Kubikmeter_N',
    'Lyzeum_N',
    'Manschette_N',
    'Mißklang_N',
    'Mulatte_N',
    'Naturschönheit_N',
    'Ohrwurm_N',
    'Pachtzins_N',
    'packen_V',
    'Rezeption_N',
    'Schmiere_N',
    'Seminar_N',
    'Sensation_N',
    'Spielball_N',
    'Tier_N',
    'Titel_N',
    'Tragfähigkeit_N',
    'Truppenteil_N',
    'überspannen_V',
    'Unentschlossenheit_N',
    'verbauen_V',
    'vergönnen_V',
    'voranstellen_V',
    'vorliegen_V',
    'vorweisen_V',
    'weitgreifend_A',
    'zersetzen_V'
    ]

## Re-parse with Stanza

In [ ]:
out_folder_reparsed = f'./SemEval_ger/corpus{corp_no}/reparsed/'
os.makedirs(os.path.dirname(out_folder_reparsed), exist_ok=True)

In [ ]:
stanza.download('de')
nlp = stanza.Pipeline(
        'de',
        processors='tokenize,mwt,pos,lemma,depparse',
        use_gpu=True,
        verbose=False,
        tokenize_no_ssplit=True
    )

In [ ]:
for corp_no in corp_nos:
    token_df = pd.read_csv(f'./SemEval_ger/corpus{corp_no}/token/ger{corp_no}.csv')

    reparsed_sents = [] 

    i = 0
    for sent in token_df['sent']:
        doc = nlp(sent)
        for s in doc.sentences:
            lines = []
            lines.append(f'<s id=ger{corp_no}_{i}>')
            for w in s.words:
                lines.append(
                    f"{w.text}\t{w.lemma}\t{w.upos}\t{w.id}\t{w.head}\t{w.deprel}"
                )
            lines.append("</s>")
            reparsed_sents.append("\n".join(lines))
            i += 1

    # Save reparsed sentences to file
    with open(f'{out_folder_reparsed}/ccoha{corp_no}_reparsed.txt', 'w') as f:
        f.write("\n\n".join(reparsed_sents))

## Check, quantify the mismatches

In [ ]:
# MAIN FUNCTION TO CHECK FOR MISMACHES

# Create 1 dictionary to store mismatch for all selected_lemmas
report_mismatches = {}

for corp_no in [1, 2]:
    report_mismatches[corp_no] = {}

    # Lemma org file
    org_parsed_file = f'./SemEval_ger/corpus{corp_no}/lemma/ger{corp_no}.csv'
    org_df = pd.read_csv(org_parsed_file)

    # Reparsed file
    reparsed_file = f'./SemEval_ger/corpus{corp_no}/reparsed/ger{corp_no}_reparsed.txt'
    with open(reparsed_file, 'r') as f:
        reparsed_content = f.read()
        # Separate the sents
        reparsed_sents = reparsed_content.strip().split("\n\n")
        reparsed_sent_count = len(reparsed_sents)

    # Check no of sents
    org_df['sent'] = org_df['sent'].fillna('') # Fillna because there are some empty lines
    org_sents = org_df['sent'].tolist() 
    org_sent_count = len(org_sents)

    if org_sent_count != reparsed_sent_count:
        report_mismatches[corp_no]['Mismatched numbers of sentences'] = f"org={org_sent_count}, reparsed={reparsed_sent_count}"
        continue
    
    # Statistics for each lemma
    for selected_lemma in selected_lemmas:
        selected_lemma_base = selected_lemma.rsplit('_', 1)[0] # In other datasets than English, there is no _POS in the target

        # Whole file statistics:
        mismatch_sent = 0
        org_miss_lemma_count = 0
        reparsed_miss_lemma_count = 0
        report_mismatches[corp_no][selected_lemma] = []
        
        # Count selected_lemma_base in file
        pattern = rf'\b{re.escape(selected_lemma_base)}\b'

        org_lemma_count = org_df['sent'].str.count(pattern).sum()
        if org_lemma_count == 0: # Avoid division by zero error if the lemma is not found
            org_lemma_count = 1
        
        # Individual sentence check
        for i in range(org_sent_count):
            org_sent = org_sents[i]
            reparsed_sent = reparsed_sents[i]

            # Count selected_lemma_base occurrences in original sentence
            org_count = len(re.findall(pattern, org_sent))

            # Count selected_lemma occurrences in reparsed sentence
            stanza_lemma, stanza_pos = convert_org_stanza(selected_lemma)
            stanza_format = f'\t{stanza_lemma}\t{stanza_pos}\t'
            reparsed_count = reparsed_sent.count(stanza_format)

            # Return stanza tags for the selected_lemma
            stanza_tags = return_stanza_parsed_tags(reparsed_sent, selected_lemma)

            if org_count != reparsed_count:
                mismatch_sent += 1
                (report_mismatches[corp_no][selected_lemma].append(
                    f"Mismatch sentence:{i}, "
                    f"org={org_count}, "
                    f"reparsed={reparsed_count}, "
                    f"org_sent='{org_sent}, "
                    f"stanza_pos={stanza_tags}"))
                
                if org_count >= reparsed_count:
                    reparsed_miss_lemma_count += (org_count - reparsed_count)
                elif org_count < reparsed_count:
                    org_miss_lemma_count += (reparsed_count - org_count)

        # Whole file statistics:
        if mismatch_sent != 0:
            report_mismatches[corp_no][selected_lemma].append(f"Total mismatched sentences: {mismatch_sent} ({mismatch_sent/org_sent_count*100:.2f}%)")
            report_mismatches[corp_no][selected_lemma].append(f"Total missing lemma in original file (compared to org): {org_miss_lemma_count} ({org_miss_lemma_count/ (org_lemma_count)*100:.2f}%)")
            report_mismatches[corp_no][selected_lemma].append(f"Total missing lemma in reparsed file (compared to org): {reparsed_miss_lemma_count} ({reparsed_miss_lemma_count/org_lemma_count*100:.2f}%)")

In [ ]:
report_mismatches

In [ ]:
with open('./SemEval_ger/mismatch_report.txt', 'w') as f:
    # Write the report_mismatches dictionary to the file beautifully
    for corp_no in report_mismatches:
        f.write(f"Corpus {corp_no}:\n")
        for selected_lemma in report_mismatches[corp_no]:
            f.write(f"Lemma: {selected_lemma}\n")
            for mismatch in report_mismatches[corp_no][selected_lemma]:
                f.write(f"{mismatch}\n")
            f.write("\n")

## Fix the mismatches

In [ ]:
file_paths = [
    './SemEval_ger/corpus1/reparsed/ger1_reparsed.txt',
    './SemEval_ger/corpus2/reparsed/ger2_reparsed.txt'  
]

org_paths = [
    './semeval2020_ulscd_ger/corpus1/lemma/ger1.txt',
    './semeval2020_ulscd_ger/corpus2/lemma/ger2.txt'
]

### Noun vs PROPN

In [ ]:
N_PROPN_patterns = [
    '\tAbgesang\tPROPN',
    '\tAckergerät\tPROPN',
    '\tArmenhaus\tPROPN',
    '\tAusnahmegesetz\tPROPN',
    '\tDynamik\tPROPN',
    '\tEinreichung\tPROPN',
    '\tEintagsfliege\tPROPN',
    '\tEngpaß\tPROPN',
    '\tEntscheidung\tPROPN',
    '\tFestspiel\tPROPN',
    '\tFrechheit\tPROPN',
    '\tFuß\tPROPN',
    '\tGesichtsausdruck\tPROPN',
    '\tKnotenpunkt\tPROPN',
    '\tKubikmeter\tPROPN',
    '\tLyzeum\tPROPN',
    '\tManschette\tPROPN',
    '\tMißklang\tPROPN',
    '\tMulatte\tPROPN',
    '\tNaturschönheit\tPROPN',
    '\tOhrwurm\tPROPN',
    '\tPachtzins\tPROPN',
    '\tRezeption\tPROPN',
    '\tSchmiere\tPROPN',
    '\tSeminar\tPROPN',
    '\tSensation\tPROPN',
    '\tSpielball\tPROPN',
    '\tTier\tPROPN',
    '\tTitel\tPROPN',
    '\tTragfähigkeit\tPROPN',
    '\tTruppenteil\tPROPN',
    '\tUnentschlossenheit\tPROPN',
    ]

for file_path in file_paths:
    content = open_txt(file_path)
    for pattern in N_PROPN_patterns:
        count = search_in_txt(content, pattern)
        if count > 0:
            print(pattern, count)

In [ ]:
# Replacements
N_PROPN_patterns_replacement ={
    '\tArmenhaus\tPROPN': '\tArmenhaus\tNOUN',
    '\tAbgesang\tPROPN': '\tAbgesang\tNOUN',
    '\tAckergerät\tPROPN': '\tAckergerät\tNOUN',
    '\tDynamik\tPROPN': '\tDynamik\tNOUN',
    '\tEinreichung\tPROPN': '\tEinreichung\tNOUN',
    '\tEintagsfliege\tPROPN': '\tEintagsfliege\tNOUN',
    '\tEngpaß\tPROPN': '\tEngpaß\tNOUN',
    '\tEntscheidung\tPROPN': '\tEntscheidung\tNOUN',
    '\tFestspiel\tPROPN': '\tFestspiel\tNOUN',
    '\tFrechheit\tPROPN': '\tFrechheit\tNOUN',
    '\tFuß\tPROPN': '\tFuß\tNOUN',
    '\tKubikmeter\tPROPN': '\tKubikmeter\tNOUN',
    '\tLyzeum\tPROPN': '\tLyzeum\tNOUN',
    '\tManschette\tPROPN': '\tManschette\tNOUN',
    '\tMißklang\tPROPN': '\tMißklang\tNOUN',
    '\tMulatte\tPROPN': '\tMulatte\tNOUN',
    '\tOhrwurm\tPROPN': '\tOhrwurm\tNOUN',
    '\tPachtzins\tPROPN': '\tPachtzins\tNOUN',
    '\tRezeption\tPROPN': '\tRezeption\tNOUN',
    '\tSchmiere\tPROPN': '\tSchmiere\tNOUN',
    '\tSeminar\tPROPN': '\tSeminar\tNOUN',
    '\tSensation\tPROPN': '\tSensation\tNOUN',
    '\tSpielball\tPROPN': '\tSpielball\tNOUN',
    '\tTier\tPROPN': '\tTier\tNOUN',
    '\tTitel\tPROPN': '\tTitel\tNOUN',
    '\tTragfähigkeit\tPROPN': '\tTragfähigkeit\tNOUN',
    '\tTruppenteil\tPROPN': '\tTruppenteil\tNOUN'
    }

for file_path in file_paths:
    content = open_txt(file_path)
    for org, repl in N_PROPN_patterns_replacement.items():
        content = replace_in_txt(content, org, repl)

    save_to_txt(content, file_path)

### ADJ vs VERB

In [ ]:
ADJ_V_patterns = [
    '\tabgebrüht\tVERB',
    '\tweitgreifend\tVERB',
    ]

for file_path in file_paths:
    content = open_txt(file_path)
    for pattern in ADJ_V_patterns:
        count = search_in_txt(content, pattern)
        if count > 0:
            print(pattern, count)

### ADJ vs NOUN

In [ ]:
ADJ_N_patterns = [
    '\tNaturschönheit\tA',
    '\tTragfähigkeit\tA',
    '\tUnentschlossenheit\tA',
    '\tFrechheit\tA',
    ]

for file_path in file_paths:
    content = open_txt(file_path)
    for pattern in ADJ_N_patterns:
        count = search_in_txt(content, pattern)
        if count > 0:
            print(pattern, count)

### Spelling

In [ ]:
spelling_patterns = [
    '\tEngpass\t', # ß -> ss
    '\tMissklang_N\t',
    '\tFuss\t',
    
    '\tabgebruht\t', # No umlaut
    '\tAckergerat\t',
    '\tNaturschonheit\t',
    '\tTragfahigkeit\t',
    '\tuberspannen\t',
    '\tvergonnen\t',

    '\tabgesang\tNOUN', # No capitalisation
    '\tackergerät\tNOUN',
    '\tarmenhaus\tNOUN',
    '\tausnahmegesetz\tNOUN',
    '\tdynamik\tNOUN',
    '\teinreichung\tNOUN',
    '\teintagsfliege\tNOUN',
    '\tengpaß\tNOUN',
    '\tentscheidung\tNOUN',
    '\tfestspiel\tNOUN',
    '\tfrechheit\tNOUN',
    '\tfuß\tNOUN',
    '\tgesichtsausdruck\tNOUN',
    '\tknotenpunkt\tNOUN',
    '\tkubikmeter\tNOUN',
    '\tlyzeum\tNOUN',
    '\tmanschette\tNOUN',
    '\tmißklang\tNOUN',
    '\tmulatte\tNOUN',
    '\tnaturschönheit\tNOUN',
    '\tohrwurm\tNOUN',
    '\tpachtzins\tNOUN',
    '\trezeption\tNOUN',
    '\tschmiere\tNOUN',
    '\tseminar\tNOUN',
    '\tsensation\tNOUN',
    '\tspielball\tNOUN',
    '\ttier\tNOUN',
    '\ttitel\tNOUN',
    '\ttragfähigkeit\tNOUN',
    '\ttruppenteil\tNOUN',
    '\tunentschlossenheit\tNOUN',

    # Individual words
    '\tAbgeſang\t',
    '\tAckergeraͤth\t',
    '\tArmenhauſe\t',
    '\tArmenhäuſer\t',

    '\tarticulerien\t',
    '\tarticulieren\t',
    '\tartikuliren\t',
    '\tartikulir\t',
    'articuliren\tarticulir\t',
    '\tartikulienen\t',
    'articulirt\tarticulirt\tPROPN',

    '\tAusnahmegeſetz\t',
    '\tAusnahmsgeſetz\t',

    '\tausſpannen\t',
    '\tausſpinnen\t',
    '\tbeimiſchen\t',

    '\tEngpaͤſſe\t',
    '\tEngpaſſ\t',
    '\tEntſcheidung\t',
    '\tFeſtſpiel\t',
    '\tFuͤß\t',
    '\tFuſ\t',
    '\tFus\t',
    '\tFüßse\t',

    '\tGeſichtsausdruck\t',
    '\tKubikmèter\t',
    '\tCubikmeter\t',
    '\tLyceum\t',
    '\tManſchette\t',
    '\tMißklaͤnge\t',
    '\tMisklang\t',
    '\tNaturſchönheit\t',
    '\tPachtzinſ\t',
    '\tReception\t',
    '\tSchmirn\t',
    '\tSchmir\t',
    '\tSenſation\t',
    '\tThiere\t',
    '\tThiers\t',
    '\tThieren\t',
    '\tTruppentheil\t',
    '\tuͤberſpannen\t',
    '\tuͤberſpinnen\t',
    '\tUnentſchloſſenheit\t',
    '\tvoranſtellen\t',
    '\tvorweiſen\t',
    '\tzerſetzen\t',

    '\tabgebrühen\t',
    '\tüberspinnen\t',
    ]

for file_path in file_paths:
    print('CORPUS')
    content = open_txt(file_path)
    for pattern in spelling_patterns:
        count = search_in_txt(content, pattern)
        if count > 0:
            print(pattern, count)

In [ ]:
# Replacements
spelling_patterns_replacement ={
    '\tEngpass\t': '\tEngpaß\t',
    '\tabgesang\tNOUN': '\tAbgesang\tNOUN',
    '\teinreichung\tNOUN': '\tEinreichung\tNOUN',
    '\tentscheidung\tNOUN': '\tEntscheidung\tNOUN',
    '\tfestspiel\tNOUN': '\tFestspiel\tNOUN',
    '\tsensation\tNOUN': '\tSensation\tNOUN',
    '\ttier\tNOUN': '\tTier\tNOUN',
    '\tAckergerat\t': '\tAckergerät\t',
    '\tTragfahigkeit\t': '\tTragfähigkeit\t',
    '\tfuß\tNOUN': '\tFuß\tNOUN',
    '\tknotenpunkt\tNOUN': '\tKnotenpunkt\tNOUN',
    '\trezeption\tNOUN': '\tRezeption\tNOUN',
    '\tseminar\tNOUN': '\tSeminar\tNOUN',
    # New
    '\tAbgeſang\t': '\tAbgesang\t',
    '\tAckergeraͤth\t': '\tAckergerät\t',
    '\tArmenhauſe\t': '\tArmenhaus\t',
    '\tArmenhäuſer\t': '\tArmenhaus\t',
    '\tarticulerien\t': '\tartikulieren\t',
    '\tarticulieren\t': '\tartikulieren\t',
    '\tartikuliren\t': '\tartikulieren\t',
    '\tartikulir\t': '\tartikulieren\t',
    'articuliren\tarticulir\t': 'articuliren\tartikulieren\t',
    '\tartikulienen\t': '\tartikulieren\t',
    'articulirt\tarticulirt\tPROPN': 'articulirt\tartikulieren\tVERB',
    '\tAusnahmegeſetz\t': '\tAusnahmegesetz\t',
    '\tAusnahmsgeſetz\t': '\tAusnahmegesetz\t',
    '\tausſpannen\t': '\tausspannen\t',
    '\tausſpinnen\t': '\tausspannen\t',

    '\tbeimiſchen\t': '\tbeimischen\t',
    '\tEngpaͤſſe\t': '\tEngpaß\t',
    '\tEngpaſſ\t': '\tEngpaß\t',
    '\tEntſcheidung\t': '\tEntscheidung\t',
    '\tFeſtſpiel\t': '\tFestspiel\t',

    '\tFuͤß\t': '\tFuß\t',
    '\tFuſ\t': '\tFuß\t',
    '\tFus\t': '\tFuß\t',
    '\tFüßse\t': '\tFuß\t',

    '\tGeſichtsausdruck\t': '\tGesichtsausdruck\t',
    '\tKubikmèter\t': '\tKubikmeter\t',
    '\tCubikmeter\t': '\tKubikmeter\t',
    '\tLyceum\t': '\tLyzeum\t',
    '\tManſchette\t': '\tManschette\t',
    '\tMißklaͤnge\t': '\tMißklang\t',
    '\tMisklang\t': '\tMißklang\t',
    '\tNaturſchönheit\t': '\tNaturschönheit\t',
    '\tPachtzinſ\t': '\tPachtzins\t',
    '\tReception\t': '\tRezeption\t',
    '\tSchmirn\t': '\tSchmiere\t',
    '\tSchmir\t': '\tSchmiere\t',
    '\tSenſation\t': '\tSensation\t',
    '\tThiere\t': '\tTier\t',
    '\tThiers\t': '\tTier\t',
    '\tThieren\t': '\tTier\t',
    '\tTruppentheil\t': '\tTruppenteil\t',
    '\tuͤberſpannen\t': '\tüberspannen\t',
    '\tuͤberſpinnen\t': '\tüberspannen\t',
    '\tUnentſchloſſenheit\t': '\tUnentschlossenheit\t',
    '\tvoranſtellen\t': '\tvoranstellen\t',
    '\tvorweiſen\t': '\tvorweisen\t',
    '\tzerſetzen\t': '\tzersetzen\t',

    '\tabgebrühen\t': '\tabgebrüht\t',
    '\tüberspinnen\t': '\tüberspannen\t',
    
    }

for file_path in file_paths:
    content = open_txt(file_path)
    for org, repl in spelling_patterns_replacement.items():
        content = replace_in_txt(content, org, repl)

    save_to_txt(content, file_path)

## Merge 2 corpus

In [ ]:
merge_output =  pilot_folder / 'merged_corpus/'
os.makedirs(merge_output, exist_ok=True)
corpus_1 = pilot_folder / 'corpus1' / 'reparsed/'
corpus_2 = pilot_folder / 'corpus2' / 'reparsed/'
corpus_file_pattern = r'ger(\d+)_reparsed.txt'

In [ ]:
file1_path = corpus_1 / f'ger1_reparsed.txt'
file2_path = corpus_2 / f'ger2_reparsed.txt'

merged_output_folder = merge_output
corpus_1_folder = merged_output_folder / '1'
corpus_2_folder = merged_output_folder / '2'

os.makedirs(corpus_1_folder, exist_ok=True)
os.makedirs(corpus_2_folder, exist_ok=True)

# Copy file1 and file 2 to merged folder
shutil.copy(file1_path, corpus_1_folder)
shutil.copy(file2_path, corpus_2_folder)

## For SemEval: Unified the POS of the target

In [ ]:
SemEval_file_paths = [
    './SemEval_ger_SemEval/corpus1/reparsed/ger1_reparsed.txt',
    './SemEval_ger_SemEval/corpus2/reparsed/ger2_reparsed.txt'  
]

In [ ]:
SemEval_lemmas = [
    'abbauen',
    'abdecken',
    'abgebrüht',
    'Abgesang',
    'Ackergerät',
    'Armenhaus',
    'artikulieren',
    'aufrechterhalten',
    'Ausnahmegesetz',
    'ausspannen',
    'beimischen',
    'Dynamik',
    'Einreichung',
    'Eintagsfliege',
    'Engpaß',
    'Entscheidung',
    'Festspiel',
    'Frechheit',
    'Fuß',
    'Gesichtsausdruck',
    'Knotenpunkt',
    'Kubikmeter',
    'Lyzeum',
    'Manschette',
    'Mißklang',
    'Mulatte',
    'Naturschönheit',
    'Ohrwurm',
    'Pachtzins',
    'packen',
    'Rezeption',
    'Schmiere',
    'Seminar',
    'Sensation',
    'Spielball',
    'Tier',
    'Titel',
    'Tragfähigkeit',
    'Truppenteil',
    'überspannen',
    'Unentschlossenheit',
    'verbauen',
    'vergönnen',
    'voranstellen',
    'vorliegen',
    'vorweisen',
    'weitgreifend',
    'zersetzen'
    ]

In [ ]:
import re

for file_path in SemEval_file_paths:
    content = open_txt(file_path)

    for lemma in SemEval_lemmas:
        # Match: \tlemma\t(ANY_POS)\t and replace the POS by TAR
        content = re.sub(
            rf'\t{re.escape(lemma)}\t[^\t]+\t',   # any POS between tabs
            f'\t{lemma}\tTAR\t',
            content
        )

    save_to_txt(content, file_path)


## Merge 2 corpus SemEval

In [ ]:
merge_output =  pilot_folder_SemEval / 'merged_corpus/'
os.makedirs(merge_output, exist_ok=True)
corpus_1 = pilot_folder_SemEval / 'corpus1' / 'reparsed/'
corpus_2 = pilot_folder_SemEval / 'corpus2' / 'reparsed/'
corpus_file_pattern = r'ger(\d+)_reparsed.txt'

In [ ]:
file1_path = corpus_1 / f'ger1_reparsed.txt'
file2_path = corpus_2 / f'ger2_reparsed.txt'

merged_output_folder = merge_output
corpus_1_folder = merged_output_folder / '1'
corpus_2_folder = merged_output_folder / '2'

os.makedirs(corpus_1_folder, exist_ok=True)
os.makedirs(corpus_2_folder, exist_ok=True)

# Copy file1 and file 2 to merged folder
shutil.copy(file1_path, corpus_1_folder)
shutil.copy(file2_path, corpus_2_folder)